# Credit Risk – Enterprise LDA
## Step 1 – Imports & Seed

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
import joblib, time, os
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
np.random.seed(42)
print('Libraries loaded')

Libraries loaded


## Step 2 – Data Loading

In [2]:
df = pd.read_csv('data/credit_risk/credit_risk_dataset.csv').dropna()
print(df.shape)
df.head()

(28638, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## Step 3 – Preprocessing Pipeline

In [3]:
y = df['loan_status']
X = df.drop('loan_status', axis=1)
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer()), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (22910, 11), Test: (5728, 11)


## Step 4 – Algorithm Pipeline Construction

In [4]:
pipeline = Pipeline([('pre', preprocessor), ('clf', LinearDiscriminantAnalysis())])
print('Pipeline:', pipeline)

Pipeline: Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer()),
                                                                  ('sc',
                                                                   StandardScaler())]),
                                                  ['person_age',
                                                   'person_income',
                                                   'person_emp_length',
                                                   'loan_amnt', 'loan_int_rate',
                                                   'loan_percent_income',
                                                   'cb_person_cred_hist_length']),
                                                 ('cat',
                                                  Pipeline(steps=[('imp',
               

## Step 5 – Baseline Dummy Benchmark

In [5]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
dummy_acc = accuracy_score(y_test, dummy.predict(X_test))
print(f'Dummy accuracy: {dummy_acc:.4f}')

Dummy accuracy: 0.7833


## Step 6 – Training & Convergence Monitoring

In [6]:
pipeline.fit(X_train, y_train)
print('Training complete')

Training complete


## Step 7 – Test Set Evaluation

In [7]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred))

Test Accuracy: 0.8588
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      4487
           1       0.73      0.55      0.63      1241

    accuracy                           0.86      5728
   macro avg       0.81      0.75      0.77      5728
weighted avg       0.85      0.86      0.85      5728



## Step 8 – Visualization (Confusion Matrix)

In [8]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=80)
plt.show()
print('Confusion matrix saved.')

Confusion matrix saved.


## Step 9 – Cross-Validation Stability (3-Fold)

In [9]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

CV Accuracy: 0.8649 ± 0.0022


## Step 10 – Production Serialization

In [10]:
os.makedirs('production_models', exist_ok=True)
pipeline.fit(X_train, y_train)
out_path = f'production_models/lda_credit_pipeline.joblib'
joblib.dump(pipeline, out_path, compress=3)
size_kb = os.path.getsize(out_path) / 1024
print(f'Pipeline saved to {out_path} ({size_kb:.1f} KB)')

Pipeline saved to production_models/lda_credit_pipeline.joblib (3.2 KB)


## Step 11 – Latency Smoke Test

In [11]:
X_sample = X_test.iloc[:100]
times = []
for _ in range(100):
    t0 = time.perf_counter()
    pipeline.predict(X_sample)
    times.append((time.perf_counter() - t0) * 1000)
avg_lat = np.mean(times)
print(f'Avg latency: {avg_lat:.2f} ms')
assert avg_lat < 50.0, f'Latency {avg_lat:.2f}ms exceeds 50ms SLA'
print('✅ Latency SLA passed')

Avg latency: 3.63 ms
✅ Latency SLA passed


## Step 12 – Advanced Analysis / Hyperparameter Tuning

In [12]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
results = []
for solver in ['svd', 'lsqr', 'eigen']:
    try:
        pip_tmp = Pipeline([('pre', preprocessor),
                            ('clf', LinearDiscriminantAnalysis(solver=solver))])
        sc = cross_val_score(pip_tmp, X, y, cv=3, scoring='accuracy').mean()
        results.append({'solver': solver, 'cv_accuracy': sc})
        print(f'  solver={solver}: {sc:.4f}')
    except Exception as e:
        print(f'  solver={solver} failed: {e}')
results_df = pd.DataFrame(results)
print(results_df)

  solver=svd: 0.8585
  solver=lsqr: 0.8585


  solver=eigen failed: 
All the 3 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 860, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/python/03_Custom_addons/ML/.venv/lib/python3.11/site-packages/sklearn/pipeline.py", line 645, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "/home/python/03_Custom_addons/ML/.venv/l

## Executive Decision Matrix

| Metric | Dummy Baseline | LDA Model |
|--------|---------------|----------------|
| Accuracy | (dummy_acc) | (test_acc) |
| Strategy | most_frequent | LDA |
| CV Stability | N/A | cv_mean ± cv_std |
| Latency | N/A | avg_lat ms |
| Prod Ready | ❌ | ✅ |

**Recommendation**: Deploy LDA model — outperforms baseline with acceptable latency.